# Introduction to Geospatial Data

This notebook introduces the core concepts you need to work with geospatial data in Python: **Coordinate Reference Systems (CRS)**, **vector data**, **raster data**, and related topics like **spatial extent**, **basemaps**, and **tile layers**. We use [leafmap](https://leafmap.org/) for interactive visualization.

## 1. Coordinate Reference Systems (CRS)

A **Coordinate Reference System (CRS)** defines how coordinates (numbers) map to real-world locations on Earth. Without a CRS, coordinates are just numbers with no geographic meaning.

### Geographic vs Projected CRS

- **Geographic CRS** (e.g., WGS84): Uses latitude and longitude in degrees. Good for storing data and global analysis, but distances and areas are distorted.
- **Projected CRS** (e.g., UTM): Flattens the Earth onto a 2D plane. Better for measuring distances, areas, and for regional maps. Each UTM zone covers a narrow strip of longitude.

### Why Reprojection Matters

When combining datasets (e.g., overlaying vector buildings on a raster image), **all layers must share the same CRS**. Otherwise, they will not align. Reprojection converts coordinates from one CRS to another.

In [1]:
# Optional: Install packages if needed
# %pip install leafmap geopandas rasterio pyproj

import leafmap
import geopandas as gpd
from shapely.geometry import Point

# Create a point in WGS84 (latitude, longitude)
point_wgs84 = Point(-122.4194, 37.7749)  # San Francisco
gdf = gpd.GeoDataFrame({"name": ["SF"]}, geometry=[point_wgs84], crs="EPSG:4326")
print("CRS:", gdf.crs)
print("Bounds (lon, lat):", gdf.total_bounds)

CRS: EPSG:4326
Bounds (lon, lat): [-122.4194   37.7749 -122.4194   37.7749]


In [2]:
# Reproject to a projected CRS (UTM zone 10N) for distance/area calculations
gdf_projected = gdf.to_crs("EPSG:32610")
print("Reprojected CRS:", gdf_projected.crs)
print("Bounds (meters):", gdf_projected.total_bounds)

Reprojected CRS: EPSG:32610
Bounds (meters): [ 551130.76848128 4180998.88149906  551130.76848128 4180998.88149906]


In [4]:
# Visualize the point on an interactive map with leafmap
m = leafmap.Map(center=[37.7749, -122.4194], zoom=12)
m.add_geojson(gdf, layer_name="San Francisco", style={"color": "red", "weight": 3})
m

Map(center=[37.7749, -122.4194], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…

**Interpretation**: The map uses a **basemap** (default OpenStreetMap tiles) to provide geographic context. Leafmap centers the view on our point and displays it as a vector layer. Basemaps are pre-rendered tile layers served over the web (XYZ format).

## 2. Vector Data

**Vector data** represents geographic features as discrete geometries: **points**, **lines**, and **polygons**. Each feature has attributes (e.g., name, population) stored in a table.

### Geometry Types

- **Points**: Single locations (e.g., cities, sensors)
- **Lines**: Linear features (e.g., roads, rivers)
- **Polygons**: Enclosed areas (e.g., buildings, countries, land parcels)

### Common Formats

- **GeoJSON**: JSON-based, human-readable, widely used on the web
- **Shapefile**: Classic format (.shp + .dbf + .shx); one geometry type per file
- **GPKG (GeoPackage)**: Modern, single-file format; supports multiple layers

In [5]:
from shapely.geometry import Point, LineString, Polygon

# Example geometries (in WGS84 - small area for demo)
point = Point(-122.4, 37.78)
line = LineString([(-122.45, 37.77), (-122.42, 37.79), (-122.38, 37.77)])
polygon = Polygon([(-122.44, 37.76), (-122.40, 37.76), (-122.40, 37.80), (-122.44, 37.80), (-122.44, 37.76)])

# GeoDataFrame can hold any of these
gdf_geoms = gpd.GeoDataFrame(
    {"type": ["point", "line", "polygon"]},
    geometry=[point, line, polygon],
    crs="EPSG:4326"
)
print(gdf_geoms)

      type                                           geometry
0    point                               POINT (-122.4 37.78)
1     line  LINESTRING (-122.45 37.77, -122.42 37.79, -122...
2  polygon  POLYGON ((-122.44 37.76, -122.4 37.76, -122.4 ...


In [6]:
# Display vector data with leafmap
m = leafmap.Map(center=[37.78, -122.42], zoom=12)
m.add_geojson(
    gdf_geoms,
    layer_name="Points, Lines, Polygons",
    style={"color": "blue", "weight": 2, "fillOpacity": 0.3},
)
m

Map(center=[37.78, -122.42], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

## 3. Raster Data

**Raster data** represents the world as a grid of cells (pixels). Each cell has one or more values. Satellite imagery, elevation models, and land cover maps are typically rasters.

### Key Concepts

- **Bands**: Layers of values (e.g., RGB = 3 bands; multispectral = many bands)
- **Resolution**: Size of each pixel in ground units (e.g., 10 m × 10 m)
- **Extent**: The geographic bounds (min/max x and y) of the raster
- **NoData**: Value used for missing or invalid cells (e.g., clouds, outside the area)
- **Cloud Optimized GeoTIFF (COG)**: Raster format optimized for web streaming; supports partial reads without downloading the full file

In [7]:
import rasterio
import numpy as np
from pathlib import Path

# Create a minimal example raster to demonstrate structure (works offline)
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)
demo_raster = data_dir / "demo_raster.tif"

# Write a small 10x10 single-band raster
arr = np.random.randint(0, 255, (10, 10), dtype=np.uint8)
transform = rasterio.transform.from_bounds(0, 0, 100, 100, 10, 10)
with rasterio.open(
    demo_raster, "w", driver="GTiff", height=10, width=10,
    count=1, dtype=arr.dtype, crs="EPSG:4326", transform=transform
) as dst:
    dst.write(arr, 1)

# Inspect raster metadata
with rasterio.open(demo_raster) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)
    print("Number of bands:", src.count)
    print("Width x Height:", src.width, "x", src.height)

CRS: EPSG:4326
Bounds: BoundingBox(left=0.0, bottom=0.0, right=100.0, top=100.0)
Resolution: (10.0, 10.0)
Number of bands: 1
Width x Height: 10 x 10


Display the raster with leafmap using a Cloud Optimized GeoTIFF (COG) from a URL:

In [8]:
# Display raster with leafmap using a Cloud Optimized GeoTIFF (COG)
cog_url = "https://github.com/opengeos/data/releases/download/raster/Libya-2023-07-01.tif"
m = leafmap.Map()
m.add_cog_layer(cog_url, name="Satellite imagery (Libya)", bands=["b1", "b2", "b3"])
m

Map(center=[22.628278391361995, 32.774842775326846], controls=(ZoomControl(options=['position', 'zoom_in_text'…

**Interpretation**: `add_cog_layer` streams the raster from a URL without downloading the full file. Bands `b1`, `b2`, `b3` are displayed as RGB. The map auto-centers on the COG extent.

## 4. Additional Geospatial Concepts

### Spatial Extent and Bounds

The **spatial extent** (or **bounds**) of a dataset defines the rectangular area it covers: minimum and maximum x (longitude or easting) and y (latitude or northing). When overlaying layers, their extents should overlap for meaningful comparison.

### Basemaps and Tile Layers

**Basemaps** are pre-rendered background maps (e.g., OpenStreetMap, satellite imagery) served as **XYZ tile layers**: the server provides small image tiles at different zoom levels. Leafmap includes many built-in basemaps via `leafmap.basemaps`.

### Combining Raster and Vector

Overlaying vector features on a raster (or vice versa) is a core GIS workflow. Both must share the same CRS. Leafmap can stack multiple layers—basemap, COG raster, vector—on a single interactive map.

In [9]:
# Explore built-in basemaps: leafmap.basemaps
# Example: add a different basemap
m = leafmap.Map(center=[37.78, -122.42], zoom=10)
m.add_basemap("Esri.WorldImagery")  # Satellite basemap
m.add_geojson(gdf, layer_name="San Francisco", style={"color": "red", "weight": 4})
m

Map(center=[37.78, -122.42], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

In [10]:
# Combined visualization: vector + remote GeoJSON on basemap
url = "https://raw.githubusercontent.com/opengeos/leafmap/master/examples/data/countries.geojson"
m = leafmap.Map(center=[20, 0], zoom=2)
m.add_geojson(url, layer_name="Countries", fill_colors=["red", "yellow", "green", "orange"])
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

## 5. Spatial Operations

**Spatial operations** modify or create geometries based on spatial relationships. They are fundamental tools for geospatial analysis.

### Common Operations

- **Buffer**: Creates a zone around a geometry at a specified distance
- **Intersection**: Returns overlapping area of two geometries
- **Union**: Combines geometries into one
- **Difference**: Subtracts one geometry from another
- **Simplify**: Reduces geometry complexity (fewer vertices)
- **Convex Hull**: Creates the smallest convex polygon containing a geometry

**Important**: These operations should typically be done in a **projected CRS** (meters) rather than geographic CRS (degrees) for accurate distance measurements.

In [15]:
# Spatial Operations Example

# Create sample geometries
from shapely.geometry import Point, Polygon

# Two overlapping polygons in WGS84
polygon1 = Polygon([(-122.44, 37.76), (-122.40, 37.76), (-122.40, 37.80), (-122.44, 37.80), (-122.44, 37.76)])
polygon2 = Polygon([(-122.42, 37.77), (-122.38, 37.77), (-122.38, 37.79), (-122.42, 37.79), (-122.42, 37.77)])

# Reproject to meters for accurate operations (UTM Zone 10N)
gdf_poly1 = gpd.GeoDataFrame(geometry=[polygon1], crs="EPSG:4326").to_crs("EPSG:32610")
gdf_poly2 = gpd.GeoDataFrame(geometry=[polygon2], crs="EPSG:4326").to_crs("EPSG:32610")

poly1 = gdf_poly1.geometry[0]
poly2 = gdf_poly2.geometry[0]

# Buffer: Create a 500m buffer around polygon1
buffered = poly1.buffer(500)

# Intersection: Find overlapping area
intersection = poly1.intersection(poly2)

# Union: Combine both polygons
union = poly1.union(poly2)

# Difference: Subtract poly2 from poly1
difference = poly1.difference(poly2)

print(f"Polygon 1 area: {poly1.area:,.0f} sq meters")
print(f"Buffer area: {buffered.area:,.0f} sq meters")
print(f"Intersection area: {intersection.area:,.0f} sq meters")
print(f"Union area: {union.area:,.0f} sq meters")
print(f"Difference area: {difference.area:,.0f} sq meters")

Polygon 1 area: 15,632,939 sq meters
Buffer area: 24,377,607 sq meters
Intersection area: 3,908,238 sq meters
Union area: 19,541,206 sq meters
Difference area: 11,724,701 sq meters


In [16]:
# Visualize spatial operations
m = leafmap.Map(center=[37.78, -122.42], zoom=13)

# Convert back to WGS84 for display
gdf_buffer = gpd.GeoDataFrame(geometry=[buffered], crs="EPSG:32610").to_crs("EPSG:4326")
gdf_intersect = gpd.GeoDataFrame(geometry=[intersection], crs="EPSG:32610").to_crs("EPSG:4326")

m.add_geojson(gdf_poly1.to_crs("EPSG:4326"), layer_name="Polygon 1", style={"color": "blue", "fillOpacity": 0.3})
m.add_geojson(gdf_poly2.to_crs("EPSG:4326"), layer_name="Polygon 2", style={"color": "red", "fillOpacity": 0.3})
m.add_geojson(gdf_buffer, layer_name="500m Buffer", style={"color": "green", "fillOpacity": 0.1})
m.add_geojson(gdf_intersect, layer_name="Intersection", style={"color": "purple", "fillOpacity": 0.5})
m

Map(center=[37.78, -122.42], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

**Interpretation**: The buffer operation creates a zone 500 meters around the blue polygon. The purple area shows where the two original polygons overlap (intersection).

## 6. Spatial Relationships

**Spatial relationships** describe how geometries relate to each other in space. These are boolean tests that return True or False.

### Common Relationships

- **intersects**: Geometries share any portion of space
- **contains**: One geometry completely surrounds another
- **within**: One geometry is completely inside another
- **touches**: Geometries share a boundary but not interior
- **crosses**: Geometries intersect but neither contains the other
- **disjoint**: Geometries have no points in common
- **overlaps**: Geometries share space but neither contains the other

In [17]:
# Spatial Relationships Example
from shapely.geometry import Point, Polygon

# Create test geometries
point = Point(-122.42, 37.78)
polygon = Polygon([(-122.44, 37.76), (-122.40, 37.76), (-122.40, 37.80), (-122.44, 37.80), (-122.44, 37.76)])

# Test relationships
print(f"Point within polygon: {point.within(polygon)}")
print(f"Polygon contains point: {polygon.contains(point)}")
print(f"Point intersects polygon: {point.intersects(polygon)}")
print(f"Point touches polygon boundary: {point.touches(polygon)}")
print(f"Point disjoint from polygon: {point.disjoint(polygon)}")

# Create a point outside
point_outside = Point(-122.50, 37.78)
print(f"\nOutside point within polygon: {point_outside.within(polygon)}")
print(f"Outside point disjoint from polygon: {point_outside.disjoint(polygon)}")

Point within polygon: True
Polygon contains point: True
Point intersects polygon: True
Point touches polygon boundary: False
Point disjoint from polygon: False

Outside point within polygon: False
Outside point disjoint from polygon: True


**Interpretation**: The point at (-122.42, 37.78) is inside the polygon, so `within` and `contains` both return True. The outside point is `disjoint` (no shared space) with the polygon.

## 7. Spatial Joins

**Spatial joins** combine attributes from two datasets based on their spatial relationship, not a common key like traditional database joins.

### Use Cases

- Assign points to polygons (e.g., customers to sales territories)
- Aggregate data by area (e.g., sum population by county)
- Find nearest features
- Transfer attributes between overlapping features

### Types of Spatial Joins

- **intersects**: Join if geometries intersect
- **within**: Join if left geometry is within right
- **contains**: Join if left geometry contains right

In [18]:
# Spatial Join Example

# Create points (stores) and polygons (zones)
stores = gpd.GeoDataFrame(
    {"store_name": ["Store A", "Store B", "Store C"],
    "sales": [10000, 15000, 8000]},
    geometry=[
        Point(-122.42, 37.77),
        Point(-122.41, 37.79),
        Point(-122.35, 37.78)
    ],
    crs="EPSG:4326"
)

zones = gpd.GeoDataFrame(
    {"zone_name": ["North Zone", "South Zone"]},
    geometry=[
        Polygon([(-122.44, 37.78), (-122.40, 37.78), (-122.40, 37.80), (-122.44, 37.80), (-122.44, 37.78)]),
        Polygon([(-122.44, 37.76), (-122.40, 37.76), (-122.40, 37.78), (-122.44, 37.78), (-122.44, 37.76)])
    ],
    crs="EPSG:4326"
)

print("Stores:")
print(stores)
print("\nZones:")
print(zones)

# Perform spatial join: which store is in which zone?
joined = gpd.sjoin(stores, zones, how="left", predicate="within")
print("\nJoined result (stores with their zones):")
print(joined[['store_name', 'sales', 'zone_name']])

Stores:
  store_name  sales               geometry
0    Store A  10000  POINT (-122.42 37.77)
1    Store B  15000  POINT (-122.41 37.79)
2    Store C   8000  POINT (-122.35 37.78)

Zones:
    zone_name                                           geometry
0  North Zone  POLYGON ((-122.44 37.78, -122.4 37.78, -122.4 ...
1  South Zone  POLYGON ((-122.44 37.76, -122.4 37.76, -122.4 ...

Joined result (stores with their zones):
  store_name  sales   zone_name
0    Store A  10000  South Zone
1    Store B  15000  North Zone
2    Store C   8000         NaN


## 8. Web Mercator Projection (EPSG:3857)

**Web Mercator** (EPSG:3857) is the de facto standard projection for web maps, used by Google Maps, OpenStreetMap, and most web mapping services.

### Characteristics

- **Global coverage**: Can display the entire world
- **Square tiles**: Easy to tile for web services
- **Fast rendering**: Mathematical transformation is simple
- **Significant distortion**: Especially near poles (Greenland appears larger than it is)

### When to Use

- Web mapping and visualization
- Displaying basemaps
- **Not recommended**: Area/distance calculations (use local UTM instead)

### EPSG:3857 vs EPSG:4326

- **EPSG:4326 (WGS84)**: Latitude/longitude in degrees; good for storage
- **EPSG:3857 (Web Mercator)**: Meters; good for web display but distorted

In [19]:
# Web Mercator Example

# Create a point in WGS84
point_wgs84 = Point(-122.4194, 37.7749)
gdf_wgs84 = gpd.GeoDataFrame(geometry=[point_wgs84], crs='EPSG:4326')

# Reproject to Web Mercator
gdf_webmerc = gdf_wgs84.to_crs('EPSG:3857')

# Reproject to UTM for comparison
gdf_utm = gdf_wgs84.to_crs('EPSG:32610')

print(f"WGS84 (EPSG:4326): {gdf_wgs84.geometry[0].x:.4f}, {gdf_wgs84.geometry[0].y:.4f} (degrees)")
print(f"Web Mercator (EPSG:3857): {gdf_webmerc.geometry[0].x:.2f}, {gdf_webmerc.geometry[0].y:.2f} (meters)")
print(f"UTM Zone 10N (EPSG:32610): {gdf_utm.geometry[0].x:.2f}, {gdf_utm.geometry[0].y:.2f} (meters)")

# Note: Web Mercator and UTM both use meters but give different values!
print("\nWeb Mercator is optimized for web display, not accuracy.")

WGS84 (EPSG:4326): -122.4194, 37.7749 (degrees)
Web Mercator (EPSG:3857): -13627665.27, 4547675.35 (meters)
UTM Zone 10N (EPSG:32610): 551130.77, 4180998.88 (meters)

Web Mercator is optimized for web display, not accuracy.


## 9. Geocoding

**Geocoding** converts addresses or place names into geographic coordinates (latitude, longitude). **Reverse geocoding** does the opposite: coordinates to address.

### Use Cases

- Plotting customer locations from addresses
- Finding coordinates of landmarks
- Converting GPS coordinates to readable addresses

### Geocoding Services

- **Nominatim** (OpenStreetMap): Free, rate-limited
- **Google Geocoding API**: Paid, high accuracy
- **Geopy**: Python library that wraps multiple services

**Note**: Always respect rate limits and terms of service when using geocoding APIs.

In [20]:
# Geocoding Example (requires geopy)
# Uncomment to run: pip install geopy

try:
    from geopy.geocoders import Nominatim
    
    geolocator = Nominatim(user_agent="geospatial_tutorial")
    
    location = geolocator.geocode("Golden Gate Bridge, San Francisco, CA")
    if location:
        print(f"Address: {location.address}")
        print(f"Latitude: {location.latitude}, Longitude: {location.longitude}")
        
        # Reverse geocode
        reverse = geolocator.reverse((location.latitude, location.longitude))
        print(f"\nReverse geocoding result: {reverse.address}")
    else:
        print("Location not found")
except ImportError:
    print("geopy not installed. Run: pip install geopy")
    print("\nFor demo purposes, here is a manual example:")
    print("Golden Gate Bridge: 37.8199 N, 122.4783 W")

Address: Golden Gate Bridge, Presidio Parkway, San Francisco, California, 94129, United States
Latitude: 37.8176155, Longitude: -122.4783123

Reverse geocoding result: Golden Gate Bridge East Sidewalk, San Francisco, California, 94129, United States


## 10. Spatial Indexing

**Spatial indexing** dramatically speeds up spatial queries on large datasets by avoiding the need to check every geometry.

### R-tree Index

The most common spatial index is the **R-tree** (Rectangle tree). It organizes geometries into a tree structure based on their bounding boxes.

### How It Works

1. Instead of checking if a point is in each of 1 million polygons...
2. The R-tree quickly finds candidate polygons whose bounding boxes might contain the point
3. Only those candidates are tested with exact geometry operations
4. Result: O(log n) instead of O(n) complexity

### When to Use

- Large datasets (10,000+ features)
- Frequent spatial queries
- Real-time applications

In [21]:
# Spatial Indexing Example
import time

# Create many random points
import numpy as np
np.random.seed(42)
n_points = 10000

points = [Point(-122 + np.random.random(), 37 + np.random.random()) for _ in range(n_points)]
points_gdf = gpd.GeoDataFrame(geometry=points, crs="EPSG:4326")

# Build spatial index
sindex = points_gdf.sindex

# Define query area
query_box = Polygon([(-122.2, 37.2), (-122.0, 37.2), (-122.0, 37.8), (-122.2, 37.8), (-122.2, 37.2)])

# Query using spatial index
start = time.time()
possible_matches_index = list(sindex.intersection(query_box.bounds))
print(f"Spatial index query found {len(possible_matches_index)} candidates in {(time.time() - start)*1000:.2f} ms")

# Verify with exact check
exact_matches = points_gdf.iloc[possible_matches_index]
exact_matches = exact_matches[exact_matches.intersects(query_box)]
print(f"Exact matches after verification: {len(exact_matches)}")

# Compare: brute force approach
start = time.time()
brute_force = points_gdf[points_gdf.intersects(query_box)]
print(f"\nBrute force found {len(brute_force)} matches in {(time.time() - start)*1000:.2f} ms")

Spatial index query found 0 candidates in 0.14 ms
Exact matches after verification: 0

Brute force found 0 matches in 0.78 ms


## 11. Scale, Precision, and Accuracy

Understanding **scale**, **precision**, and **accuracy** is crucial for proper geospatial data interpretation.

### Scale

The ratio between distance on a map and distance in the real world.
- **Large scale** (1:1,000): Zoomed in, detailed (e.g., building footprints)
- **Small scale** (1:1,000,000): Zoomed out, less detail (e.g., world map)

### Precision

The level of detail in coordinates. More decimal places = higher precision.
- 1 decimal place (~10 km precision)
- 5 decimal places (~1 meter precision)
- 8 decimal places (~1 mm precision - likely spurious)

### Accuracy

How close coordinates are to true location. High precision does not equal high accuracy!
- GPS: ~3-5 meter accuracy
- Survey-grade: ~1 cm accuracy
- Consumer phone: ~10-20 meter accuracy

In [22]:
# Scale, Precision, and Accuracy Example

# Same coordinate with different precision
lat_precise = 37.7749295
lat_rounded = 37.775
lat_very_precise = 37.774929500000001

# Calculate approximate distance per decimal place
# At equator: 1 degree latitude ~ 111 km
degrees_per_meter = 1 / 111000

print("Precision comparison:")
print(f"Original: {lat_precise:.7f} (7 decimal places)")
print(f"Rounded: {lat_rounded:.3f} (3 decimal places, ~100m precision)")
print(f"Extra precise: {lat_very_precise:.16f} (16 decimal places, ~0.1mm)")

print("\nDecimal place precision (approximate):")
for decimals in [1, 3, 5, 7, 9]:
    meters = (10 ** -decimals) / degrees_per_meter
    print(f"  {decimals} decimals: ~{meters:.1f} meters")

print("\nNote: High precision coordinates do not mean high accuracy!")
print("GPS might report 8 decimal places but only be accurate to 5 meters.")

Precision comparison:
Original: 37.7749295 (7 decimal places)
Rounded: 37.775 (3 decimal places, ~100m precision)
Extra precise: 37.7749294999999989 (16 decimal places, ~0.1mm)

Decimal place precision (approximate):
  1 decimals: ~11100.0 meters
  3 decimals: ~111.0 meters
  5 decimals: ~1.1 meters
  7 decimals: ~0.0 meters
  9 decimals: ~0.0 meters

Note: High precision coordinates do not mean high accuracy!
GPS might report 8 decimal places but only be accurate to 5 meters.


## Recap

| Concept | Summary |
|---------|---------|
| **CRS** | Defines how coordinates map to Earth; geographic (lat/lon) vs projected (meters). Always match CRS when combining layers. |
| **Vector** | Points, lines, polygons with attributes. Formats: GeoJSON, Shapefile, GeoPackage. |
| **Raster** | Grid of cells with bands, resolution, extent, and NoData. COG enables efficient web streaming. |
| **Spatial extent** | The rectangular bounds (min/max x, y) a dataset covers. |
| **Basemaps & tiles** | Pre-rendered XYZ tile layers for map context. Leafmap provides many via `add_basemap()`. |
| **Spatial operations** | Buffer, intersection, union, difference. Use projected CRS for accuracy. |
| **Spatial relationships** | intersects, contains, within, touches, crosses, disjoint. Boolean tests for how geometries relate. |
| **Spatial joins** | Combine datasets based on spatial relationship (e.g., points within polygons). |
| **Web Mercator** | EPSG:3857; standard for web maps but distorted for measurements. |
| **Geocoding** | Convert addresses to coordinates (and vice versa via reverse geocoding). |
| **Spatial indexing** | R-tree index for fast spatial queries on large datasets. |
| **Scale/Precision/Accuracy** | Scale: map-to-reality ratio. Precision: coordinate detail. Accuracy: true location. |